In [0]:
df = spark.read.table("ecommerce_analytics.bronze.sales_orders")
df.display()

### # Array of Array - 
[["AVpfPEx61cnluZ0-gyT9","34"],["AVpfuJ4pilAPnD_xhDyM","98"],["AVpe6jFBilAPnD_xQxO2","60"],["AVpfIODe1cnluZ0-eg35","49"]]
[["AVpfdBS41cnluZ0-lBIj","88"]]

### # Array of Struct -
[{"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},{"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},{"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}]

### # Array of Struct -
[{"promo_disc":0.03,"promo_id":"0","promo_item":"AVpfMVD-ilAPnD_xW6bu","promo_qty":"2"}]


### Three most important functions-
from json -string > structured column

explode function - convert one row to multiple rows

explode to outer - same as explode but handle if data is null

example - 19476252 AVpfPEx61cnluZ0-gyT9 34
19476252 AVpfuJ4pilAPnD_xhDyM 98
19476252 AVpe6jFBilAPnD_xQxO2 60
19476252 AVpfIODe1cnluZ0-eg35 49

In [0]:
df.printSchema()

In [0]:
df.select("ordered_products").display()

[{"curr":"USD","id":"AVpfuJ4pilAPnD_xhDyM","name":"Rony LBT-GPX555 Mini-System with Bluetooth and NFC","price":"993","promotion_info":null,"qty":"3","unit":"pcs"},{"curr":"USD","id":"AVpe6jFBilAPnD_xQxO2","name":"Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface","price":"218","promotion_info":null,"qty":"3","unit":"pcs"},{"curr":"USD","id":"AVpfIODe1cnluZ0-eg35","name":"Cyber-shot DSC-WX220 Digital Camera (Black)","price":"448","promotion_info":null,"qty":"2","unit":"pcs"}]
--> StructType and Structfield --> ArrayType

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType, StructType, StructField, ArrayType

ordered_products = ArrayType(
    StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", StringType()),
        StructField("promotion_info", StringType()),
        StructField("qty", StringType()),
        StructField("unit", StringType())
    ])
)
df_parsed = df.withColumn("ordered_products", from_json(col("ordered_products"), ordered_products))
df_parsed.display()

In [0]:
from pyspark.sql.functions import explode_outer
df_exploded_op = df_parsed.withColumn("ordered_products", explode_outer("ordered_products"))
df_exploded_op.display()


In [0]:
df_ordered_products = df_exploded_op.select("customer_id", "customer_name",
"order_number",col("ordered_products.id").alias("order_products_id"),col("ordered_products.name").alias("product_name"),"ordered_products.price",
"ordered_products.curr",
col("ordered_products.qty").alias("ordered_products_qty"),
"ordered_products.unit")
df_ordered_products.display()

In [0]:
#Define promo schema
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, ArrayType

promo_schema = ArrayType(
    StructType([
        StructField("promo_disc", DoubleType()),
        StructField("promo_id", StringType()),
        StructField("promo_item", StringType()),
        StructField("promo_qty", StringType())
    ])
)

In [0]:
# Step 2: Parse JSON
from pyspark.sql.functions import from_json, col

df_parsed = df.withColumn(
    "promo_info",
    from_json(col("promo_info"), promo_schema)
)
df_parsed


In [0]:
from pyspark.sql.functions import explode_outer

df_exploded = df_parsed.withColumn(
    "promo_info",
    explode_outer("promo_info")
)
df_exploded.display()


In [0]:
from pyspark.sql.types import IntegerType, DecimalType

df_promo_silver = df_exploded_promo.select(
    col("customer_id"),
    col("order_number"),
    col("product_id"),   # from previous silver

    col("promo_info.promo_id").alias("promo_id"),
    col("promo_info.promo_item").alias("promo_item_id"),

    col("promo_info.promo_disc")
        .alias("promo_discount"),

    col("promo_info.promo_qty")
        .alias("promo_quantity")
)
df_promo_silver.display()

In [0]:
df_promo_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_analytics.silver.order_promotions")